In [1]:
import re
from collections import Counter, defaultdict
from datasets import load_dataset

print("=" * 60)
print("SMART NEXT-WORD PREDICTOR")
print("=" * 60)


# ------------------------------------------------------------
# 1. LOAD WIKITEXT-2 DATASET
# ------------------------------------------------------------

print("\nLoading WikiText-2 dataset...")

dataset = load_dataset(
    "Salesforce/wikitext",
    "wikitext-2-raw-v1"
)

print("\nWikiText-2 Dataset Loaded Successfully")
print("Train Samples:", len(dataset["train"]))
print("Validation Samples:", len(dataset["validation"]))
print("Test Samples:", len(dataset["test"]))


# ------------------------------------------------------------
# 2. PREPARE CORPUS
# ------------------------------------------------------------

print("\nPreparing WikiText-2 corpus...")

# Use training data
text = " ".join(dataset["train"]["text"])

# Convert to lowercase
# Keep only alphabetic words
tokens = re.findall(r"[a-zA-Z]+", text.lower())

print("Total Words:", len(tokens))
print("Vocabulary Size:", len(set(tokens)))


# ------------------------------------------------------------
# 3. UNIGRAM MODEL
# ------------------------------------------------------------

print("\nBuilding Unigram Frequency Table...")

unigram_counts = Counter(tokens)

print("Unigram Frequency Table Created")


# ------------------------------------------------------------
# 4. BIGRAM MODEL
# ------------------------------------------------------------

print("Building Bigram Frequency Table...")

bigram_counts = Counter(
    (tokens[i], tokens[i + 1])
    for i in range(len(tokens) - 1)
)

print("Bigram Frequency Table Created")


# ------------------------------------------------------------
# 5. TRIGRAM MODEL
# ------------------------------------------------------------

print("Building Trigram Frequency Table...")

trigram_counts = Counter(
    (tokens[i], tokens[i + 1], tokens[i + 2])
    for i in range(len(tokens) - 2)
)

print("Trigram Frequency Table Created")


# ------------------------------------------------------------
# 6. CREATE NEXT-WORD TABLES
# ------------------------------------------------------------

next_word_counts = defaultdict(Counter)

for (word1, word2, word3), count in trigram_counts.items():

    next_word_counts[(word1, word2)][word3] = count


bigram_next_word_counts = defaultdict(Counter)

for (word1, word2), count in bigram_counts.items():

    bigram_next_word_counts[word1][word2] = count


print("\nLanguage Model Tables Created Successfully")


# ------------------------------------------------------------
# 7. PROBABILITY FUNCTIONS
# ------------------------------------------------------------

def unigram_probability(word):

    return unigram_counts[word] / len(tokens)


def bigram_probability(word1, word2):

    denominator = unigram_counts[word1]

    if denominator == 0:
        return 0

    return bigram_counts[(word1, word2)] / denominator


def trigram_probability(word1, word2, word3):

    denominator = bigram_counts[(word1, word2)]

    if denominator == 0:
        return 0

    return trigram_counts[
        (word1, word2, word3)
    ] / denominator


print("Probability Calculation Completed")


# ------------------------------------------------------------
# 8. NEXT-WORD PREDICTION
# ------------------------------------------------------------

def predict_next_words(sentence, top_n=5):

    words = re.findall(
        r"[a-zA-Z]+",
        sentence.lower()
    )

    if len(words) == 0:
        return []


    # --------------------------------------------------------
    # TRIGRAM PREDICTION
    # --------------------------------------------------------

    if len(words) >= 2:

        word1 = words[-2]
        word2 = words[-1]

        candidates = next_word_counts.get(
            (word1, word2),
            {}
        )

        if candidates:

            predictions = []

            for word3 in candidates:

                probability = trigram_probability(
                    word1,
                    word2,
                    word3
                )

                predictions.append(
                    (word3, probability)
                )


            predictions.sort(
                key=lambda x: x[1],
                reverse=True
            )

            return predictions[:top_n]


    # --------------------------------------------------------
    # BIGRAM BACKOFF
    # --------------------------------------------------------

    last_word = words[-1]

    candidates = bigram_next_word_counts.get(
        last_word,
        {}
    )

    if candidates:

        predictions = []

        for next_word in candidates:

            probability = bigram_probability(
                last_word,
                next_word
            )

            predictions.append(
                (next_word, probability)
            )


        predictions.sort(
            key=lambda x: x[1],
            reverse=True
        )

        return predictions[:top_n]


    # --------------------------------------------------------
    # UNIGRAM BACKOFF
    # --------------------------------------------------------

    predictions = []

    for word, count in unigram_counts.most_common(top_n):

        probability = unigram_probability(word)

        predictions.append(
            (word, probability)
        )

    return predictions


# ------------------------------------------------------------
# 9. DISPLAY PREDICTIONS
# ------------------------------------------------------------

def display_prediction(sentence):

    print("\n" + "-" * 60)

    print("Input Sentence:")
    print(sentence)

    predictions = predict_next_words(
        sentence,
        top_n=5
    )

    print("\nTop Predictions:")

    if not predictions:

        print("No prediction available")

        return


    for rank, (word, probability) in enumerate(
        predictions,
        start=1
    ):

        print(
            f"{rank}. {word:<20}"
            f"Probability: {probability:.6f}"
        )


# ------------------------------------------------------------
# 10. TEST SENTENCES
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("TESTING NEXT-WORD PREDICTIONS")
print("=" * 60)


test_sentences = [
    "the",
    "in the",
    "of the",
    "one of",
    "according to"
]


for sentence in test_sentences:

    display_prediction(sentence)


# ------------------------------------------------------------
# 11. USER INPUT
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("CUSTOM NEXT-WORD PREDICTION")
print("=" * 60)

user_sentence = input(
    "\nEnter a sentence: "
)

display_prediction(user_sentence)


# ------------------------------------------------------------
# 12. COMPLETION
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("NEXT-WORD PREDICTION COMPLETED")
print("=" * 60)

SMART NEXT-WORD PREDICTOR

Loading WikiText-2 dataset...



WikiText-2 Dataset Loaded Successfully
Train Samples: 36718
Validation Samples: 3760
Test Samples: 4358

Preparing WikiText-2 corpus...
Total Words: 1694394
Vocabulary Size: 61884

Building Unigram Frequency Table...
Unigram Frequency Table Created
Building Bigram Frequency Table...
Bigram Frequency Table Created
Building Trigram Frequency Table...
Trigram Frequency Table Created

Language Model Tables Created Successfully
Probability Calculation Completed

TESTING NEXT-WORD PREDICTIONS

------------------------------------------------------------
Input Sentence:
the

Top Predictions:
1. first               Probability: 0.017053
2. th                  Probability: 0.010744
3. song                Probability: 0.008817
4. game                Probability: 0.007341
5. same                Probability: 0.006974

------------------------------------------------------------
Input Sentence:
in the

Top Predictions:
1. united              Probability: 0.031326
2. s                   Probability


Enter a sentence:  machine learning



------------------------------------------------------------
Input Sentence:
machine learning

Top Predictions:
1. that                Probability: 0.133333
2. curve               Probability: 0.093333
3. the                 Probability: 0.080000
4. to                  Probability: 0.080000
5. about               Probability: 0.066667

NEXT-WORD PREDICTION COMPLETED
